`28/09/2026`

# TCC Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator

plt.style.use('Ltx.mplstyle')

In [ ]:
figs_folder = "./figures/"

In [ ]:
Perc7_DKBLUE_1 = '#340498'
Perc7_PURPUR_2 = '#6c00a8'
Perc7_VIOLET_3 = '#a31d99'
Perc7_PINKSH_4 = '#ca4678'
Perc7_AMBERS_5 = '#e97257'
Perc7_ORANGE_6 = '#fa9f3a'
Perc7_YELLOW_7 = '#f8db24'

Perc7 = [Perc7_DKBLUE_1, Perc7_PURPUR_2, Perc7_VIOLET_3, Perc7_PINKSH_4, Perc7_AMBERS_5, Perc7_ORANGE_6, Perc7_YELLOW_7]

Perc5_DKBLUE_1 = '#340498'
Perc5_PURPLE_2 = '#7e03a7'
Perc5_PINKSH_3 = '#b83289'
Perc5_ROSSEE_4 = '#de6063'
Perc5_ORANGE_5 = '#f8983d'

Perc5 = [Perc5_DKBLUE_1, Perc5_PURPLE_2, Perc5_PINKSH_3, Perc5_ROSSEE_4, Perc5_ORANGE_5]
Perc5_reverse = Perc5[::-1]


In [ ]:
# Physical units
cm = 1 / 2.54

# Report dimensions
WIDTH = {
    "single": 15 * cm,
    "double": 15 * cm,
    "triple": 15 * cm,
    "half": 7.2 * cm,
}

HEIGHT = {
    "single": 9.0 * cm,
    "double": 6.5 * cm,
    "triple": 5.5 * cm,
    "half": 5.0 * cm,
}

def figure(kind="single", aspect=0.65, **kwargs):
    """Create a report-sized figure.

    aspect = height / width
    """
    return plt.subplots(
        figsize=(WIDTH[kind], WIDTH[kind] * aspect),
        layout="constrained",
        **kwargs,
    )

In [ ]:
def apply_axis_style(ax, tick_len=5):
    # if ax.get_xscale() != "log":
    #     ax.xaxis.set_minor_locator(AutoMinorLocator())
    # if ax.get_yscale() != "log":
    #     ax.yaxis.set_minor_locator(AutoMinorLocator())
    # ax.tick_params(which='both', direction='in', right=True, top=True)
    # ax.tick_params(which='major', length=tick_len)
    try:
        ax.get_yticklabels()[0].set_verticalalignment("bottom")
        ax.get_yticklabels()[-1].set_verticalalignment("top")
    except (IndexError, AttributeError):
        pass
    try:
        ax.get_xticklabels()[0].set_horizontalalignment("left")
        ax.get_xticklabels()[-1].set_horizontalalignment("right")
    except (IndexError, AttributeError):
        pass

In [ ]:
from emerald.classical.dynamics import StaticDynamics
from emerald.potentials import Coulomb, Morse, MorseSoftCoulomb, SoftCoulomb


In [ ]:
## Figure 1

msc = MorseSoftCoulomb(alpha=1)
sc  = SoftCoulomb(alpha=1)
cou = Coulomb()
mor = Morse(D=1, beta=1)

fig_potentials, (ax_pots_1, ax_pots_2) = figure("double", 0.8, nrows=2)

r_grid = np.linspace(-10, 10, 500)

# Panel a)

ax_pots_1.set_title("(a)", loc='left', weight='bold')
for p, pot in enumerate([cou, sc, mor, msc]):
    ax_pots_1.plot(r_grid, pot(r_grid), c=Perc5_reverse[p], label=f"{pot.print(short=True, latex=True)}")
ax_pots_1.legend(loc='upper left')
ax_pots_1.grid(which='major')
ax_pots_1.set_xlim(-10, 10)
ax_pots_1.set_ylim(-1.5, 1.5)
apply_axis_style(ax_pots_1)

# Panel b)

ax_pots_2.set_title("(b)", loc='left', weight='bold')
for i, _a in enumerate([0.1, 0.5, 1, 2]):
    msc = MorseSoftCoulomb(_a)
    ax_pots_2.plot(r_grid, msc(r_grid), c=Perc5_reverse[i], label=f"{msc.print(short=True, latex=True)}")
ax_pots_2.plot(r_grid, cou(r_grid), c=Perc5[0], ls='dashed', label=r"MsC $\alpha=0.00$ (Coulomb)")

ax_pots_2.legend(loc='lower right')
ax_pots_2.grid(which='major')
ax_pots_2.set_xlim(-3, 5)
ax_pots_2.set_ylim(-5, 1)
apply_axis_style(ax_pots_2)

# remaining labels
for _ax in [ax_pots_1, ax_pots_2]:
    _ax.set_xlabel("$r$")
    _ax.set_ylabel("$V(r)$", rotation=0, labelpad=10)

# whitespace on the right

axr = ax_pots_1.twinx()
axr.set_ylabel(
    r"$\mathrm{placeholder}$",
    color="none",
    rotation=0
)
axr.tick_params(
    right=False,
    labelright=False
)
axr.spines["right"].set_visible(False)

In [ ]:
fig_potentials.savefig(f"{figs_folder}Fig-Potentials.pdf")

In [ ]:
## Figure 2

mor = Morse(D=1, beta=1)

fig_ps, ax_ps = figure("single", 0.65)

r_vals = np.linspace(-0.75, 11, 20)     # shape (20,)
p_max = mor.momentum(E=0.1, r=0)
p_vals = np.linspace(-1.1*p_max, 1.1*p_max, 20)     #shape (20,)
R, P = np.meshgrid(r_vals, p_vals)  # shapes (20, 20) & (20, 20)

fake_mor = Morse(D=1, beta=0.5)
dyn = StaticDynamics(fake_mor)
X = np.column_stack((R.reshape(R.size, 1), P.reshape(P.size, 1)))   
# ^ shape (400, 2) every (r, p) permutation
Y = dyn.dstate(0, X) # shape (400, 2), maps every (r, p) position to (ṙ, ṗ)
U = Y[:, 0].reshape(R.shape)    #back to (20, 20) of ṙ values
V = Y[:, 1].reshape(R.shape)    #back to (20, 20) of ṗ values

# plot the arrows
ax_ps.quiver(R, P, U, V, angles='xy', pivot='middle', color='grey', alpha=.5, scale=40, width=0.002)


r_grid = np.linspace(-1, 11, 500)

for i, _E in enumerate([-0.9, -0.5, -0.1, 0, 0.1]):
    ax_ps.plot(*mor.phase_space(_E, 500, r2=11), label=f"$E={_E:+.2f}$")
ax_ps.legend(loc='upper right')
# ax1_1.grid(which='major')
ax_ps.set_xlim(-1, 11)
# ax2.set_ylim(-2, 2)

ax_ps.set_xlabel("$r$")
ax_ps.set_ylabel("$p$", rotation=0, labelpad=10)
# apply_axis_style(ax2)

# whitespace on the right

axr = ax_ps.twinx()
axr.set_ylabel(
    r"$\mathrm{placeholder}$",
    color="none",
    rotation=0
)
axr.tick_params(
    right=False,
    labelright=False
)
axr.spines["right"].set_visible(False)

In [ ]:
fig_ps.savefig(f"{figs_folder}Fig-Morse-Phase-Space.pdf")

In [ ]:
## Figure 3

from emerald.classical.dynamics import Dynamics
from emerald.fields import SinusoidField



mor = Morse(D=1, beta=1)
fld = SinusoidField(0.065, 0.8, 0)

fig_m_traj, (ax_traj_1, ax_traj_2, ax_traj_3) = figure("double", 1, nrows=3, gridspec_kw={"height_ratios": [1, 1, 2.5]})

static_dyn = StaticDynamics(mor)
dyn = Dynamics(mor, fld)

# times = np.linspace(0, 14.5*fld.period, 10000)
times = np.linspace(0, 120, 10000)
r0 = 1
state0 = np.array([r0, mor.momentum(E=-0.5, r=r0)])

traj_static = static_dyn.propagate(state0, t_span=(times[0], times[-1]), t_eval=times)
traj = dyn.propagate(state0, t_span=(times[0], times[-1]), t_eval=times)

ax_traj_1.plot(traj_static.time, traj_static.position, ls='dotted', c='grey')
ax_traj_1.plot(traj.time, traj.position)
ax_traj_1.set_title("(a)", loc='left', weight='bold')
ax_traj_1.set_ylabel("$r$", rotation=0, labelpad=10)
ax_traj_1.tick_params(labelbottom=False)
ax_traj_1.set_xlim(traj.time[0], traj.time[-1])
ax_traj_1.set_ylim(-1, 10)

ax_traj_2.plot(traj_static.time, traj_static.momentum, ls='dotted', c='grey')
ax_traj_2.plot(traj.time, traj.momentum)
ax_traj_2.set_title("(b)", loc='left', weight='bold')
ax_traj_2.set_ylabel("$p$", rotation=0, labelpad=10)
ax_traj_2.set_xlabel("$t$")
ax_traj_2.set_xlim(traj.time[0], traj.time[-1])
ax_traj_2.set_ylim(-2, 2)
ax_traj_2.sharex(ax_traj_1)

# add arrows indicating trajectory direction
arrows = 20
# concentrate arrows at start and end of trajectory
arrow_indices = np.concatenate((
    np.linspace(1, len(traj.time)//4, arrows//2, dtype=int),
    np.linspace(4*len(traj.time)//5, len(traj.time)-1, arrows//2, dtype=int)
))
for i in arrow_indices:
    ax_traj_3.annotate(
        '',
        xy=(traj.position[i], traj.momentum[i]),
        xytext=(traj.position[i-1], traj.momentum[i-1]),
        arrowprops={'arrowstyle': '-|>', 'color': Perc5_DKBLUE_1, 'lw': 0.9},
        size=10,
    )

ax_traj_3.set_title("(c)", loc='left', weight='bold')
ax_traj_3.plot(*mor.phase_space(-0.5, N=500), ls='dotted', c='grey', label="$F(t)=0$")
ax_traj_3.plot(traj.position, traj.momentum, label=r"$F(t)=0.065\sin(0.8 t)$")
ax_traj_3.set_ylabel("$p$", rotation=0, labelpad=10)
ax_traj_3.set_xlabel("$r$")
ax_traj_3.set_xlim(-2, 12)
ax_traj_3.legend(loc='upper right')

for _ax in (ax_traj_1, ax_traj_2, ax_traj_3):
    _ax.grid(which='major')
    apply_axis_style(_ax)

axr = ax_traj_1.twinx()
axr.set_ylabel(
    r"$\mathrm{placeholder}$",
    color="none",
    rotation=0
)
axr.tick_params(
    right=False,
    labelright=False
)
axr.spines["right"].set_visible(False)


In [ ]:
traj.metadata

In [ ]:
fig_m_traj.savefig(f"{figs_folder}Fig-Morse-Trajectory.pdf")

In [ ]:
fig, ax = plt.subplots()

fig = ax.figure
spec = ax.get_subplotspec()

ax.remove()
ax = fig.add_subplot(spec, projection='polar')

ticks = np.linspace(-np.pi, np.pi, 9)
ax.set_theta_zero_location('W')
ax.set_xticks(ticks)
ax.set_xticklabels([
    r'$\pm\pi$',
    r'$-3\pi/4$',
    r'$-\pi/2$',
    r'$-\pi/4$',
    r'$0$',
    r'$\pi/4$',
    r'$\pi/2$',
    r'$3\pi/4$',
    r''
])
ax.set_thetamin(-180)
ax.set_thetamax(180)

ax.plot(traj.angle, traj.action, lw=1)

In [ ]:
fig_m_traj.savefig(f"{figs_folder}Fig-Morse-Trajectory.pdf")


In [ ]:
fig_integrand, (ax_integ_1, ax_integ_2) = figure("double", 0.8, nrows=2)


phis = np.linspace(0, np.pi/2, 500)


for _E in [-0.7, -0.5, -0.3]:
    rm, rM = mor.turning_points(_E)
    rs = np.linspace(rm, rM, 2000)
    ax_integ_1.plot(rs, 1/mor.momentum(_E, rs), label=rf"$E = {_E:+.1f}$")
    ax_integ_1.axvline(rm, c='grey', ls='dashed')
    # ax4_1.text(
    #     rm, 1, r'$r_{-}$',
    #     ha='center', va='bottom',
    #     bbox={'facecolor': 'white', 
    #           'edgecolor': 'none',
    #           'pad': 2}
    # )

    ax_integ_1.axvline(rM, c='grey', ls='dashed')
    # ax4_1.text(
    #     rM, 1, r'$r_{+}$',
    #     ha='center', va='bottom',
    #     bbox={'facecolor': 'white', 
    #           'edgecolor': 'none',
    #           'pad': 2}
    # )

ax_integ_1.set_xlabel('$r$')
ax_integ_1.set_ylabel(r'$p^{-1}(r)$', rotation=0, labelpad=15)
ax_integ_1.set_title('(a)', loc='left', weight='bold')
ax_integ_1.legend(loc='upper left',
    bbox_to_anchor=(0.2, 1.0))
apply_axis_style(ax_integ_1)

ax_integ_1.grid(True)
ax_integ_1.set_ylim(0, 10)

for _E in [-0.7, -0.5, -0.3]:
    rm, rM = mor.turning_points(_E)
    integs = np.array([mor._inv_momentum_integrand(phi, _E, rm, rM) for phi in phis])
    ax_integ_2.plot(phis[1:-1], integs[1:-1], label=rf"$E = {_E:+.1f}$")

ax_integ_2.legend()
ax_integ_2.set_xlabel(r'$\varphi$')
ax_integ_2.set_ylabel(r'$g(\varphi)$', rotation=0, labelpad=15)
ax_integ_2.set_title('(b)', loc='left', weight='bold')
ax_integ_2.set_xticks(np.array([0, np.pi/8, np.pi/4, 3*np.pi/8, np.pi/2]))
ax_integ_2.set_xticklabels(['0', r'$\pi/8$', r'$\pi/4$', r'$3\pi/8$', r'$\pi/2$'])
ax_integ_2.set_xlim(0, np.pi/2)
ax_integ_2.set_ylim(1, 5)
ax_integ_2.grid(True)
apply_axis_style(ax_integ_2)

axr = ax_integ_2.twinx()
axr.set_ylabel(
    r"$\mathrm{placeholder}$",
    color="none",
    rotation=0
)
axr.tick_params(
    right=False,
    labelright=False
)
axr.spines["right"].set_visible(False)

In [ ]:
fig_integrand.savefig(f"{figs_folder}Fig-Momentum-Integrand.pdf")

In [ ]:
from emerald.potentials.base import Potential
import numpy as np
from scipy.integrate import cumulative_trapezoid


def angle_grid(
    potential: "Potential",
    E_range,
    M: int = 300,
    NQ: int = 500,
):
    """
    Calculate the angle grid theta(E, phi) used for interpolation.

    Parameters
    ----------
    potential : Potential
        Potential instance providing turning_points(E),
        angular_frequency(E), and _inv_momentum_integrand(...).
    E_range : array_like
        Energies defining the range of the grid. The minimum and
        maximum bound energies determine the energy interval.
    M : int, default=300
        Number of energy grid points.
    NQ : int, default=500
        Number of phi grid points.

    Returns
    -------
    E_grid : ndarray, shape (M,)
        Energy grid.
    phi_table : ndarray, shape (NQ,)
        Phi grid, spanning [0, pi/2].
    theta_grid : ndarray, shape (M, NQ)
        Angle values evaluated on the (E, phi) grid.
    """
    E_range = np.asarray(E_range, dtype=float)
    E_range = E_range[np.isfinite(E_range) & (E_range < 0)]

    if E_range.size == 0:
        raise ValueError("E_range must contain at least one bound energy.")

    E_grid = np.linspace(E_range.min(), E_range.max(), M)
    phi_table = np.linspace(0.0, np.pi / 2, NQ)

    rm = np.empty(M)
    rM = np.empty(M)
    omg = np.empty(M)

    for k, Ek in enumerate(E_grid):
        rm[k], rM[k] = potential.turning_points(Ek)
        omg[k] = potential.angular_frequency(Ek)

    with np.errstate(divide="ignore", invalid="ignore"):
        integrand = potential._inv_momentum_integrand(
            phi_table[None, :],
            E_grid[:, None],
            rm[:, None],
            rM[:, None],
        )

    # Regularize the endpoint as in the original implementation.
    integrand[:, 0] = integrand[:, 1]

    cum = cumulative_trapezoid(
        integrand,
        phi_table,
        axis=1,
        initial=0.0,
    )

    theta_grid = omg[:, None] * cum

    return E_grid, phi_table, theta_grid


In [ ]:
E_grid, phi_grid, theta_grid = angle_grid(
    mor,
    E_range=(-1, -0.0001),
)

E_mesh, phi_mesh = np.meshgrid(E_grid, phi_grid, indexing="ij")

fig_grid, ax_grid = figure("single", 0.6)

levels = np.arange(np.pi/8, np.pi, np.pi/8)

pcm = ax_grid.pcolormesh(
    phi_mesh,
    E_mesh,
    theta_grid,
    shading="auto",
    cmap="plasma",
    rasterized=True,
)

levs = ax_grid.contour(
    phi_mesh,
    E_mesh,
    theta_grid,
    levels=levels,
    colors='w',
    linewidths=1
)

level_labels = {
    np.pi/8: r"$\pi/8$",
    np.pi/4: r"$\pi/4$",
    3*np.pi/8: r"$3\pi/8$",
    np.pi/2: r"$\pi/2$",
    5*np.pi/8: r"$5\pi/8$",
    3*np.pi/4: r"$3\pi/4$",
    7*np.pi/8: r"$7\pi/8$",
}

x_pos = np.array([np.pi/8, 3*np.pi/16 ,np.pi/4, 5*np.pi/16 ,3*np.pi/8, 7*np.pi/16 ,np.pi/2])
# y_pos = np.linspace(-0.45, -0.15, 7)
y_pos = -np.logspace(np.log10(0.45), np.log10(0.15), 7)

pos = list(zip(x_pos, y_pos))

clab = ax_grid.clabel(levs, inline=True, fontsize=10, fmt=level_labels, manual=pos, )

ax_grid.set_xlabel(r"$\varphi$")
ax_grid.set_ylabel(r"$E$", rotation=0, labelpad=10)
ax_grid.set_ylim(-1, 0)

ax_grid.set_xticks(np.arange(0, np.pi/2+np.pi/8, np.pi/8))
ax_grid.set_xticklabels(['0', r'$\pi/8$', r'$\pi/4$', r'$3\pi/8$', r'$\pi/2$'])


cbar = fig_grid.colorbar(pcm, ax=ax_grid, label=r"$\theta(E,\varphi)$")
cbar.set_ticks(np.arange(0, np.pi+np.pi/8, np.pi/8))
cbar.set_ticklabels(['0', r'$\pi/8$', r'$\pi/4$', r'$3\pi/8$', r'$\pi/2$', 
                     r'$5\pi/8$', r'$3\pi/4$', r'$7\pi/8$', r'$\pi$'])


apply_axis_style(ax_grid, tick_len=7)
# set axis tick colors to white along with colorbar ticks
# set for both major and minor ticks
# leave labels black for visibility


ax_grid.tick_params(
    axis='both', which='both',
    color='white',
    labelcolor='black'
)

cbar.ax.tick_params(
    axis='y', which='both',
    color='white',
    labelcolor='black'
)


cbar.ax.get_yticklabels()[0].set_verticalalignment("bottom")
cbar.ax.get_yticklabels()[-1].set_verticalalignment("top")


In [ ]:
fig_grid.savefig(f"{figs_folder}Fig-Grid.pdf")

In [ ]:
(np.pi/2 + np.pi/4)/2, 3*np.pi/8

In [ ]:
from emerald.classical.dynamics import ClassicalSystem, Dynamics
from emerald.fields import SinusoidField

morse = Morse(D=1, beta=1)
field = SinusoidField(amplitude=0.05, 
                      ang_frequency=0.8,
                      phase=0
                    )
dyn   = Dynamics(morse, field)

E0 = -0.5
N  = 40
syst1 = ClassicalSystem.from_energy(dynamics=dyn, E=E0, N=30,
                                    momenta_signs=0, position_signs=0)
syst2 = ClassicalSystem.from_angle(dynamics=dyn, E=E0, N=30,
                                    momenta_signs=0, position_signs=0)

fig_init, (ax_init_1, ax_init_2) = figure('single', 1, nrows=2, gridspec_kw={"height_ratios": [1, 1]}) 
ax_init_1.grid(which='major', zorder=0)
# ax_init_1.grid(which='minor', alpha=0.5, zorder=1)

ax_init_1.set_title('(a)', loc='left', weight='bold')
ax_init_1.plot(*morse.phase_space(-0.5, 500), ls='dashed', c='grey', zorder=-1)
ax_init_1.scatter(*syst2.states0.T, marker='D', c=Perc5[-1], label="Ângulo", zorder=10)
ax_init_1.scatter(*syst1.states0.T, marker='x', label="Posição", zorder=11)

ax_init_1.set_xlabel(r"$r$")
ax_init_1.set_ylabel(r"$p$", rotation=0, labelpad=10)
ax_init_1.legend(loc='upper right', bbox_to_anchor=(0.995, 0.98))

ax_init_1.set_xlim(-0.6, 1.4)
ax_init_1.set_ylim(-1.2, 1.2)

ax_init_1.set_xticks(np.arange(-0.6, 1.8, 0.4))
ax_init_1.set_yticks(np.arange(-1.2, 1.6, 0.4))

apply_axis_style(ax_init_1)

fig_init = ax_init_2.figure
spec = ax_init_2.get_subplotspec()

ax_init_2.remove()
ax_init_2 = fig_init.add_subplot(spec, projection='polar')
ax_init_2.set_title('(b)', loc='left', weight='bold')

ticks = np.linspace(-np.pi, np.pi, 9)
ax_init_2.set_theta_zero_location('W')
ax_init_2.set_xticks(ticks)
ax_init_2.set_xticklabels([
    r'$\pm\pi$',
    r'$-3\pi/4$',
    r'$-\pi/2$',
    r'$-\pi/4$',
    r'$0$',
    r'$\pi/4$',
    r'$\pi/2$',
    r'$3\pi/4$',
    r''
])
ax_init_2.set_thetamin(-180)
ax_init_2.set_thetamax(180)

rm, rM = morse.turning_points(-0.5)
theta = np.linspace(-np.pi, np.pi, 500)
J = morse.action(-0.5)

ax_init_2.plot(theta, J*np.ones_like(theta), ls='dashed', c='grey')
ax_init_2.set_ylim(0, 0.6)
ax_init_2.set_yticks( np.array([0.1, 0.3, 0.6]) )
ax_init_2.set_yticklabels( ['0.1', '0.3', '0.6'] )

r = syst2.states0[:, 0]
r = r[:int(len(r)/2)]
tht = mor.angle(-0.5, r)
tht = np.append(tht, -tht)

ax_init_2.scatter(tht, J*np.ones_like(tht), marker='D', c=Perc5[-1], label="Ângulo", zorder=10)

r = syst1.states0[:, 0]
r = r[:int(len(r)/2)]
tht = mor.angle(-0.5, r)
tht = np.append(tht, -tht)

ax_init_2.scatter(tht, J*np.ones_like(tht)+0.025, marker='x', label="Posição", zorder=11)
ax_init_2.legend(bbox_to_anchor=(1.4, 1.0))


In [ ]:
J , np.sqrt(2) - 1

In [ ]:
fig_init.savefig(f"{figs_folder}Fig-Initial-Conditions.pdf")


In [ ]:
morse = Morse(D=1, beta=1)
field = SinusoidField(amplitude=0.025, 
                      ang_frequency=1.,
                      phase=0
                    )
dyn   = Dynamics(morse, field)

E0 = -0.5

syst = ClassicalSystem.from_angle(dyn, E0, 80)

pmap = syst.poincare_map(NT=2000)

In [ ]:
morse.angular_frequency(-0.5028)/field.ang_frequency

In [ ]:
%matplotlib qt5

In [ ]:
plt.scatter(pmap.angle, pmap.energy, s=5, marker=".", lw=0, c="black")
plt.xlim(-np.pi, np.pi)
# plt.ylim(-0.75, -0.2)

In [ ]:
mor.turning_points(-0.5028)

In [ ]:
mor.angle(-0.5028, -0.50474)

In [ ]:
traj = dyn.propagate(np.array([0.5, mor.momentum(-0.1250000000000002, 0.5)]), (0, 10000), np.linspace(0, 10000, 1000000))

In [ ]:
plt.plot(traj.position, traj.momentum)
# plt.plot(traj.time, traj.position)
plt.show()

In [ ]:
syst_test = ClassicalSystem(dyn, np.array([0.5, mor.momentum(-0.1250000000000002, 0.5)]))

pmap_test = syst_test.poincare_map(NT=2000)

In [ ]:
plt.scatter(pmap_test.angle, pmap_test.energy, s=5, marker=".", lw=0, c="black")
plt.xlim(-np.pi, np.pi)
# plt.ylim(-0.75, -0.2)

In [ ]:
plt.scatter(pmap_test.position, pmap_test.momentum, s=5, marker=".", lw=0, c="black")


In [ ]:
import json

with open("data/morse-poincare1.json", 'w') as filehandle:
    json.dump(pmap.to_dict(), filehandle)

In [ ]:
plt.scatter(pmap.position, pmap.momentum, s=0.25, marker=".", lw=0, c="black")
plt.xlim(0, 2)
plt.ylim(-1, 1)
# plt.ylim(-0.75, -0.2)

In [ ]:
morse.angular_frequency(-0.1250000000000002)

In [ ]:
fig, ax = plt.subplots()

fig = ax.figure
spec = ax.get_subplotspec()

ax.remove()
ax = fig.add_subplot(spec, projection='polar')

ticks = np.linspace(-np.pi, np.pi, 9)
ax.set_theta_zero_location('W')
ax.set_xticks(ticks)
ax.set_xticklabels([
    r'$\pm\pi$',
    r'$-3\pi/4$',
    r'$-\pi/2$',
    r'$-\pi/4$',
    r'$0$',
    r'$\pi/4$',
    r'$\pi/2$',
    r'$3\pi/4$',
    r''
])
ax.set_thetamin(-180)
ax.set_thetamax(180)

ax.scatter(pmap.angle, pmap.energy, lw=0, marker='.', s=.5)

In [ ]:
fig, ax = plt.subplots()

fig = ax.figure
spec = ax.get_subplotspec()

ax.remove()
ax = fig.add_subplot(spec, projection='polar')

ticks = np.linspace(-np.pi, np.pi, 9)
ax.set_theta_zero_location('W')
ax.set_xticks(ticks)
ax.set_xticklabels([
    r'$\pm\pi$',
    r'$-3\pi/4$',
    r'$-\pi/2$',
    r'$-\pi/4$',
    r'$0$',
    r'$\pi/4$',
    r'$\pi/2$',
    r'$3\pi/4$',
    r''
])
ax.set_thetamin(-180)
ax.set_thetamax(180)
# ax.set_ylim(0.2, 0.8)

ax.scatter(pmap.angle, pmap.energy, lw=0, s=0.3, marker='.')

In [ ]:
from emerald.quantum.representations import BSplineBasis

fig_bspln, (ax_spln_1, ax_spln_2) = figure("double", 0.8, nrows=2)

bps = np.array([0, 1, 2, 3, 4, 5])
k   = 3

fine_r = np.linspace(bps[0], bps[-1], 500)

basis = BSplineBasis(bps, k)

cmap = plt.get_cmap("plasma")

colors = cmap(np.linspace(0.2, 0.8, basis.number_functions))

for i in range(basis.number_functions):
    x = np.linspace(basis.knot_sequence[i], basis.knot_sequence[i+k], 500)
    y = basis.basis_functions[i](x)
    ax_spln_1.plot(x, y, c=colors[i])

    idx = np.argmax(y)  # Position of the maximum
    ha = "left" if i==0 else ("right" if i==basis.number_functions-1 else "center")
    ax_spln_1.annotate(
        rf"$B_{{{i+1}}}(r)$",
        xy=(x[idx], y[idx]),
        xytext=(0, -30),
        textcoords="offset points",
        ha=ha,
        va="bottom",
        color=colors[i],
        fontsize=10,
        bbox={'facecolor': "white", 'edgecolor': "none", 'pad': 1},
    )

ax_spln_1.set_title('(a)', loc='left', weight='bold')
ax_spln_1.scatter(bps, np.zeros_like(bps), c='black', s=20, marker='o', label=r"\textit{Breakpoints}", zorder=10)
ax_spln_1.legend()
ax_spln_1.set_xlim(bps[0]-0.05, bps[-1]+0.05)

# bps = np.array([0, 1, 2, 3, 4, 5])
bps = np.array([0, 1, 3, 11, 13, 14])

basis = BSplineBasis(bps, k)

cmap = plt.get_cmap("plasma")
colors = cmap(np.linspace(0.2, 0.8, basis.number_functions))

for i in range(basis.number_functions):
    x = np.linspace(basis.knot_sequence[i], basis.knot_sequence[i+k], 500)
    y = basis.basis_functions[i](x)
    ax_spln_2.plot(x, y, c=colors[i])

    idx = np.argmax(y)  # Position of the maximum
    ha = "left" if i==0 else ("right" if i==basis.number_functions-1 else "center")
    ax_spln_2.annotate(
        rf"$B_{{{i+1}}}(r)$",
        xy=(x[idx], y[idx]),
        xytext=(0, -20),
        textcoords="offset points",
        ha=ha,
        va="bottom",
        color=colors[i],
        fontsize=10,
        bbox={'facecolor': "white", 'edgecolor': "none", 'pad': 1},
    )

ax_spln_2.set_title('(b)', loc='left', weight='bold')
ax_spln_2.scatter(bps, np.zeros_like(bps), c='black', s=20, marker='o', label=r"\textit{Breakpoints}", zorder=10)
ax_spln_2.legend(loc='upper center')
ax_spln_2.set_xlim(bps[0]-0.1, bps[-1]+0.1)

for _ax in [ax_spln_1, ax_spln_2]:
    _ax.set_xlabel("$r$")
    _ax.set_ylabel("$B(r)$", rotation=0, labelpad=15)
    apply_axis_style(_ax)
    _ax.set_ylim(-0.025, 1)

axr = ax_spln_2.twinx()
axr.set_ylabel(
    r"$\mathrm{placeholder}$",
    color="none",
    rotation=0
)
axr.tick_params(
    right=False,
    labelright=False
)
axr.spines["right"].set_visible(False)

In [ ]:
fig_bspln.savefig(f"{figs_folder}/Fig-BSplines.pdf")

In [ ]:
morse = Morse(1, 1)

fld = SinusoidField(0.05, 0.8, 0)

dyn = Dynamics(morse, fld)

syst = ClassicalSystem.from_energy(dyn, -0.5, 400)

traj = syst.propagate(np.linspace(0, 1000, 100))

PI = traj.ionization_probabilities()
plt.plot(traj.time, PI)

PI = traj.ionization_probabilities(criterion='distance')
plt.plot(traj.time, PI)


In [ ]:
np.where(traj.energy>0, 1, 0).shape

In [ ]:
np.where(abs(traj.position) > 100, 1, 0).shape

In [ ]:
np.where((traj.energy > 0) & (abs(traj.position) > 100), 1, 0).sum(axis=1).shape

In [ ]:
traj.time.shape

In [ ]:
print("Hello")

In [ ]:
print("Bye")

In [ ]:
from emerald.quantum.coulomb_unperturbed import C_energy_levels, C_eigstate
from emerald.potentials import Coulomb

cou = Coulomb()

r = np.linspace(1.e-3, 30, 500)
V = cou(r)

plt.plot(r, V)
plt.xlim(0, 30)
plt.ylim(-1, 0)

for i in range(10):
    E = C_energy_levels(i+1)
    plt.hlines(E, *cou.turning_points(E), color='grey')
# plt.yscale('symlog')

In [ ]:
from matplotlib.gridspec import GridSpec

# Data setup
Energies = np.linspace(-0.4999, -0.05, 100)
alphas = np.linspace(1.e-4, 1.999, 100)

# Create the main figure with subplots (a) and (b)
fig_Jw = plt.Figure(figsize=(WIDTH['single'], WIDTH['single']), layout="constrained")
gs = GridSpec(3, 2, height_ratios=[1, 1, 0.5], width_ratios=[1, 0.5], figure=fig_Jw)
#plt.subplots_adjust(hspace=-0.1)  # Smaller value reduces the gap; larger value increases it.


ax_Jw_a = fig_Jw.add_subplot(gs[0, 0])
ax_Jw_a.set_title("(a)", loc='left', fontweight='bold')
ax_Jw_a.set_xticks( np.arange(-0.5, 0, 0.15) )
ax_Jw_a.set_yticks( np.arange(0, 1.25, 0.25) )
ax_Jw_a.set_ylim(0, 1)
ax_Jw_a.set_xlim(-0.5, -0.05)
ax_Jw_a.tick_params(labelbottom=False)
apply_axis_style(ax_Jw_a)

ax_Jw_b = fig_Jw.add_subplot(gs[1, 0], sharex=ax_Jw_a)
ax_Jw_b.set_title("(b)", loc='left', fontweight='bold')
ax_Jw_b.set_xticks( np.arange(-0.5, 0, 0.15) )
ax_Jw_b.set_yticks( np.arange(0., 4, 0.8) )
ax_Jw_b.set_ylim(0, 3.2)
apply_axis_style(ax_Jw_b)

ax_Jw_c = fig_Jw.add_subplot(gs[0:2, 1])
ax_Jw_c.set_title("(c)", loc='left', fontweight='bold')
ax_Jw_c.set_xlim(0, 2)

apply_axis_style(ax_Jw_c)

cou = Coulomb()
# Plot data on (a) and (b)
for i, alpha in enumerate([2.0, 1.0, 0.5, 0.1, 0]):
    if alpha > 0:
        msc = MorseSoftCoulomb(alpha)
        sc  = SoftCoulomb(alpha)
        
        omgs    = np.array([msc.angular_frequency(E) for E in Energies])
        actions = np.array([msc.action(E) for E in Energies])

        ax_Jw_a.plot(Energies, omgs, markersize=0, c=Perc5[i+1], label=f"$\\alpha = {alpha}$")
        ax_Jw_b.plot(Energies, actions, markersize=0, c=Perc5[i+1], label=f"$\\alpha = {alpha}$")

        omgs    = np.array([sc.angular_frequency(E) for E in Energies])
        actions = np.array([sc.action(E) for E in Energies])
        
        ax_Jw_a.plot(Energies, omgs, markersize=0, c=Perc5[i+1], ls='dotted')
        ax_Jw_b.plot(Energies, actions, markersize=0, c=Perc5[i+1], ls='dotted')


    else:
        omgs = np.array([cou.angular_frequency(E) for E in Energies])
        actions = np.array([cou.action(E) for E in Energies])

        ax_Jw_a.plot(Energies, omgs, markersize=0, ls="dashed", c='k', label="$\\alpha = 0$", alpha=0.5)
        ax_Jw_b.plot(Energies, actions, markersize=0, ls="dashed", c='k', label="$\\alpha = 0$", alpha=0.5)

# Add legends
ax_Jw_a.legend()
ax_Jw_b.legend()

# Labels
ax_Jw_a.set_ylabel("$\\omega$", rotation=0, labelpad=10)
ax_Jw_b.set_xlabel("$E$")
ax_Jw_b.set_ylabel("$J$", rotation=0, labelpad=10)
ax_Jw_b.invert_yaxis()

# Add text annotations for (a) and (b)
#ax1.text(-0.5, 1.05, '(a)', fontsize=12, verticalalignment='top', horizontalalignment='right')
#ax2.text(-0.5, 3.15, '(b)', fontsize=12, verticalalignment='top', horizontalalignment='right')

# Plot data on (c)

for i, E in enumerate([-0.5, -0.45, -0.4, -0.35, -0.3]):
    omegas = np.array([MorseSoftCoulomb(alpha).angular_frequency(E) for alpha in alphas])
    ax_Jw_c.plot(alphas, omegas, markersize=0, linewidth=1.5, c=Perc5[i], label=f"$E = {E:.2f}$")
    
    omegas = np.array([SoftCoulomb(alpha).angular_frequency(E) for alpha in alphas])
    ax_Jw_c.plot(alphas, omegas, markersize=0, linewidth=1.5, c=Perc5[i], ls='dotted')
    ax_Jw_c.hlines(cou.angular_frequency(E), 0, 2, ls='dashed', color='k', lw=1., alpha=0.5)
ax_Jw_c.legend()
ax_Jw_c.set_xlabel("$\\alpha$")
ax_Jw_c.set_ylabel("$\\omega$", rotation=0, labelpad=10)
ax_Jw_c.set_ylim(omegas[-1], 1)
# Adjust layout
fig_Jw


In [ ]:
fig_Jw.savefig(f"{figs_folder}Fig-MsC-Action-Freq.pdf")

In [ ]:

fig_ps, (ax_ps_a, ax_ps_b) = figure(aspect=1.1, nrows=2)

Es = np.array([-0.5, -0.25, -0.1, 0.0001, 0.1])

cmap   = plt.get_cmap('plasma')
colors = cmap(np.linspace(0.25, .75, len(Es)))
cou    = Coulomb()

r1 =-0.5 ; r2=15
msc = MorseSoftCoulomb(alpha=1)
for i, _E in enumerate(Es):
    r, p = msc.phase_space(_E, 500, r1=r1, r2=r2)
    ax_ps_a.plot(r, p, c=colors[i], label=f"$E={_E:+.2f}$")
    r, p = cou.phase_space(_E, 500, r1=0, r2=r2)
    ax_ps_a.plot(r, p, c=colors[i], ls='dashed', lw=0.5)
ax_ps_a.set_ylim(-2, 2)
ax_ps_a.set_xlim(-2, 12)
ax_ps_a.legend(loc='upper right')
ax_ps_a.set_title("(a)", loc='left', fontweight='bold')
ax_ps_a.set_xlabel(r"$r$")
ax_ps_a.set_ylabel(r"$p$", rotation=0, labelpad=10)


r1 =-15 ; r2=15
sc = SoftCoulomb(alpha=1)
for i, _E in enumerate(Es):
    r, p = sc.phase_space(_E, 500, r1=r1, r2=r2)
    ax_ps_b.plot(r, p, c=colors[i], label=f"$E={_E:+.2f}$")
    r, p = cou.phase_space(_E, 500, r1=0, r2=r2)
    ax_ps_b.plot(r, p, c=colors[i], ls='dashed', lw=0.5)
ax_ps_b.set_ylim(-2, 2)
ax_ps_b.set_xlim(-12, 12)
ax_ps_b.legend(loc='upper right')
ax_ps_b.set_title("(b)", loc='left', fontweight='bold')
ax_ps_b.set_xlabel(r"$r$")
ax_ps_b.set_ylabel(r"$p$", rotation=0, labelpad=10)




In [ ]:
fig_ps.savefig(f"{figs_folder}Fig-MsC-Phase-Space.pdf")

In [ ]:
r, p = cou.phase_space(-0.5, 500, r1=0, r2=10)
r = r[::-1] ; p = p[::-1]
plt.plot(r[1:-1], p[1:-1])
plt.ylim(-10, 10)

In [ ]:
r

In [ ]:
from emerald.classical.integrators import RK4Integrator

integ = RK4Integrator(1.e-4)

In [ ]:
msc = MorseSoftCoulomb(alpha=1)
fld = SinusoidField(amplitude=0.025,
                    ang_frequency=1.,
                    phase=0)
dyn = Dynamics(msc, fld)

pmaps = []

for _E in np.linspace(-0.6, -0.4, 9):

    syst = ClassicalSystem.from_angle(dyn, _E, 80, position_signs=1)
    pmaps.append(syst.poincare_map(200))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8), dpi=200)
for pmap in pmaps:
    # angs = msc._batch_angle(pmap.energy, pmap.position, 500, 750)
    # angs = np.sign(pmap.momentum)*angs
    # plt.scatter(angs, pmap.energy, s=0.25, marker=".", lw=0, c="black")
    plt.scatter(pmap.angle, pmap.energy, s=0.5, marker=".", lw=0, c="black")
plt.xlim(-np.pi, np.pi)
plt.ylim(-0.8, -0.2)

In [ ]:
pmap.states.shape

In [ ]:
for pmap in pmaps:
    plt.scatter(pmap.position, pmap.momentum, s=0.25, marker=".", lw=0, c="black")
plt.xlim(0, 3)
plt.ylim(-1, 1)

In [ ]:
plt.scatter(*syst3.states0.T)

In [ ]:
# Test

syst1 = ClassicalSystem.from_angle(dyn, -0.5, 80, position_signs=1)
syst2 = ClassicalSystem.from_angle(dyn, -0.4, 80, position_signs=1)

syst1.poincare_map(100)
syst2.poincare_map(100)

In [ ]:
syst1.states0.shape

In [ ]:
np.row_stack((syst1.states0, syst1.states0)).shape

In [ ]:
states01 = syst1.states0 #(80,2)
states02 = syst2.states0 #(80,2)

states03 = np.row_stack((states01, states02))

syst3 = ClassicalSystem(dyn, states03)
syst3.poincare_map(100)


In [ ]:
from emerald.quantum.coulomb_unperturbed import C_energy_levels, C_eigstate
from emerald.quantum.representations import BSplineBasis
from emerald.quantum.dynamics import Spectrum

cou = Coulomb()

bps = np.logspace(0, np.log10(5000+1), 500) - 1

cou_basis = BSplineBasis(bps, 9)

cou_spec = Spectrum.from_representation(cou_basis, cou, 'dirichlet', None)

In [ ]:
for i in range(cou_spec.n_bound):
    balm = C_energy_levels(i+1)
    spln = cou_spec.energies[i]
    dif  = np.abs(balm-spln) 
    pct  = dif/np.abs(balm)
    print(f"n = {i+1:3d} | "
          f"Balmer = {balm:<10.5f} "
          f"BSpline = {spln:<10.5f} "
          f"Difference = {dif:.3e} "
          f"Percentual = {pct*100:.3e}%")

In [ ]:
fig_wavef, ax = figure()

r_fine = np.linspace(0, 30, 1000)

A, B, C = 0.29298502, 1.29182089, 0.97409036

for i in range(3):
    ax.plot(r_fine, cou_spec.wavefunction(i)(r_fine), label=rf"$\varphi_{{{i+1}}}(r)$", c=Perc5[i+1])
# ax.plot(r_fine, B*np.exp(-A*r_fine+C)-0.01 , ls='dashed')
# ax.plot(r_fine, -C_eigstate(r_fine, i+1), ls='dashed', c='k', alpha=0.5, label='Analítico')
    rM = cou.turning_points(cou_spec.energies[i])[1]
    plt.axvline(rM, ls='dotted', c=Perc5[i+1])
    ax.annotate(
            rf"$r_+(E_{{{i+1}}})$",
            xy=(rM, 0.8),
            ha='center',
            va="center",
            color=Perc5[i+1],
            fontsize=10,
            bbox={'facecolor': "white", 'edgecolor': Perc5[i+1], 'pad': 2, 'ls': 'dotted'},)
ax.legend()
ax.set_xlim(0, 30)
ax.set_ylim(-0.4, 0.9)
# ax.set_ylim(-0.03, 0.03)

# ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useMathText=True)

ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\varphi_n(r)$", rotation=0,labelpad=10)

apply_axis_style(ax)

ax.grid()
# ax.legend()

axr = ax.twinx()
axr.set_ylabel(
    r"$\mathrm{placeholder}$",
    color="none",
    rotation=0
)
axr.tick_params(
    right=False,
    labelright=False
)
axr.spines["right"].set_visible(False)

In [ ]:
fig_wavef.savefig(f"{figs_folder}Fig-Coulomb-Wavefuncs.pdf")


In [ ]:
-1/5000 # E = -1/2n² --> n = √r/2

In [ ]:
n = np.sqrt(5000/2)
n

In [ ]:
from scipy.optimize import curve_fit

p0 = 0.2, 1, .5

def f(x, A, B, C):
    return B*np.exp(-A*x+C)

p, _ = curve_fit(f, np.linspace(rM, 200, 1000), cou_spec.wavefunction(i)(np.linspace(rM, 200, 1000)), p)

In [ ]:
p

In [ ]:
fig, ax = figure()

r_fine = np.linspace(3000, 5500, 1000)

A, B, C = 0.29298502, 1.29182089, 0.97409036

i = 90
ax.plot(r_fine, cou_spec.wavefunction(i)(r_fine), label='B-Splines')
# ax.plot(r_fine, -C_eigstate(r_fine, i+1), ls='dashed', c='k', alpha=0.5, label='Analítico')
# ax.plot(r_fine, B*np.exp(-A*r_fine+C)-0.01 , ls='dashed')
rM = cou.turning_points(cou_spec.energies[i])[1]
plt.axvline(rM, ls='dotted', c='k')

# ax.set_xlim(0, 25)
# ax.set_ylim(-0.3, 0.5)
ax.annotate(
        rf"$r_+(E_{{{i+1}}})$",
        xy=(rM, 0),
        ha='center',
        va="center",
        color='k',
        fontsize=10,
        bbox={'facecolor': "white", 'edgecolor': "k", 'pad': 2, 'ls': 'dotted'},)
# ax.set_ylim(-0.03, 0.03)

# ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useMathText=True)

ax.set_xlabel(r"$r$")
ax.set_ylabel(rf"$\varphi_{{{i+1}}}(r)$", rotation=0,labelpad=10)

apply_axis_style(ax)

ax.grid()
# ax.legend()

axr = ax.twinx()
axr.set_ylabel(
    r"$\mathrm{placeholder}$",
    color="none",
    rotation=0
)
axr.tick_params(
    right=False,
    labelright=False
)
axr.spines["right"].set_visible(False)

In [ ]:
cou_spec.wavefunction(10)(r_fine)

In [ ]:
1/C_energy_levels(i+1)

In [ ]:
from emerald.fields import SinusoidField

In [ ]:
from emerald.quantum.evolution import SplitOperatorPropagator

fund_gap = cou_spec.energies[1] - cou_spec.energies[0]

field = SinusoidField(0.005, fund_gap, 0)

prop = SplitOperatorPropagator(cou_spec, field)

In [ ]:
field.period

In [ ]:
state0 = np.zeros_like(cou_spec.energies)
state0[0] = 1

In [ ]:
times = np.arange(0, 7000.01, 1)
times.shape

In [ ]:
plt.scatter(times, field(times), s=5)
plt.xlim(0, 50)

In [ ]:
evol = prop.propagate(state0, times, record_every=5)

In [ ]:
fig_transition, ax = figure()

for i in range(3):
    ax.plot(evol.times, evol.population(i), label=fr"$c_{i+1}^2(t)$", c=Perc5[i+1])
ax.legend()
ax.set_xlim(0, 3500)
ax.set_ylim(0, 1)
ax.grid()
ax.set_xlabel(r'$t$')
ax.set_ylabel(r'$c_i^2(t)$', rotation=0, labelpad=10)

axr = ax.twinx()
axr.set_ylabel(
    r"$\mathrm{placeholder}$",
    color="none",
    rotation=0
)
axr.tick_params(
    right=False,
    labelright=False
)
axr.spines["right"].set_visible(False)

In [ ]:
fig_transition.savefig(f"{figs_folder}Fig-Level-Transition.pdf")


In [ ]:
evol.norm.shape

In [ ]:
from emerald.quantum.evolution import SplitOperatorPropagator

ionization_gap = 0.1 - cou_spec.energies[0]

field = SinusoidField(0.3, ionization_gap, 0)

prop = SplitOperatorPropagator(cou_spec, field)

In [ ]:
print(field.period)

state0 = np.zeros_like(cou_spec.energies)
state0[0] = 1

times = np.arange(0, 1000.01, 2*field.frequency)
print(times.shape)

plt.scatter(times, field(times), s=5)
plt.xlim(0, 50)

In [ ]:
evol_ion = prop.propagate(state0, times, record_every=5)

In [ ]:
plt.plot(evol_ion.times, evol_ion.ionization_probability())

In [ ]:
free_Es = cou_spec.energies[cou_spec.n_bound:]

PopE = [evol_ion.population(i)[-1] for i in range(len(free_Es))]

plt.plot(free_Es/field.ang_frequency, PopE)
[plt.vlines(i*field.ang_frequency, 0, 0.2, ls='dashed', color='grey') for i in range(100)[::2]]
plt.xlim(0, 40)
plt.ylim(0, 0.04)
plt.xlabel('$E/\\Omega$')
plt.ylabel('$P(E)$')

In [ ]:
cou_basis.number_functions

In [ ]:
from emerald.numerics.adaptive_sampling import (
    AdaptiveSampler,
    CurvatureMonitor,
    LogarithmicTailMonitor,
    SlopeMonitor,
)



In [ ]:
box_size = 5000

sc = SoftCoulomb(2.e-0)
logm = LogarithmicTailMonitor(L=sc.alpha)
# sc_fgrid = np.arange(-box_size, box_size+sc.alpha/10, sc.alpha/10)

sc_sampler = AdaptiveSampler(sc, 0, box_size, logm, int(1.e7))

bps = sc_sampler.sample(250)
bps = np.append(-bps[::-1][:-1], bps)

# plt.plot(sc_fgrid, sc(sc_fgrid))
plt.scatter(bps, sc(bps), marker='.', s=10, c='red')

plt.xlim(-5000, 5000)

In [ ]:
_as = np.array([2, 1, 0.1, 0.01, 0.001, 0.0001, 0.00001])

print(np.log10(_as))

In [ ]:
def map_logarithmic(x):
    # Map the log10 of the values, then convert back using base 10
    xp_log = np.log10([1e-5, 2])
    fp_log = np.log10([0.05, 1e-4])
    return 10**np.interp(np.log10(x), xp_log, fp_log)

In [ ]:
def map_logarithmic_msc(x):
    # Map the log10 of the values, then convert back using base 10
    xp_log = np.log10([1e-5, 2])
    fp_log = np.log10([.001, 3.5])
    return 10**np.interp(np.log10(x), xp_log, fp_log)

In [ ]:
map_logarithmic(_as)

In [ ]:
box_size = 5000

msc = MorseSoftCoulomb(.0001)
# bsln = map_logarithmic(msc.alpha)
bsln = .001
logm = LogarithmicTailMonitor(lamda=1, L=msc.alpha)

# sc_fgrid = np.arange(-box_size, box_size+sc.alpha/10, sc.alpha/10)

msc_sampler = AdaptiveSampler(msc, get_r_exact(msc.alpha, 100), box_size, logm, int(1.e6))

bps = msc_sampler.sample(500)

# plt.plot(sc_fgrid, sc(sc_fgrid))
plt.scatter(bps, msc(bps), marker='.', s=10, c='red')

plt.xlim(get_r_exact(msc.alpha, 100), .001)

In [ ]:
import numpy as np

def get_r_exact(alpha, V0: float = 100.0) -> float:
    """
    Calculates the exact position r where the Morse branch reaches value V0.
    """
    return -alpha * np.sqrt(2) * np.log(1 + np.sqrt(1 + alpha * V0))


def get_r_approx(alpha: float, V0: float = 100.0) -> float:
    """
    Calculates the high-V0 asymptotic approximation for the position r.
    Valid when alpha * V0 >> 1.
    """
    return -(alpha / np.sqrt(2)) * np.log(alpha * V0)

In [ ]:
from emerald.quantum.representations import BSplineBasis
from emerald.quantum.dynamics import Spectrum

In [ ]:
alphas = np.logspace(-4, np.log10(2), 31)

E1s_sc = []
E2s_sc = []
E3s_sc = []

E1s_msc = []
E2s_msc = []
E3s_msc = []


box_size = 5000

for _a in alphas:
    sc  = SoftCoulomb(alpha=_a)
    msc = MorseSoftCoulomb(alpha=_a)
    
    logm = LogarithmicTailMonitor(L=_a)

    sc_sampler  = AdaptiveSampler(sc, 0, box_size, logm, n_grid=int(1.e6))
    msc_sampler = AdaptiveSampler(msc, get_r_exact(msc.alpha, 100), box_size, logm, n_grid=int(1.e6))

    sc_bps = sc_sampler.sample(300)
    sc_bps = np.concatenate((-sc_bps[1:][::-1], sc_bps))

    msc_bps = msc_sampler.sample(1000)

    sc_basis  = BSplineBasis(sc_bps, 9)
    msc_basis = BSplineBasis(msc_bps, 9)

    sc_spec  = Spectrum.from_representation(sc_basis, sc, 'dirichlet', 'dirichlet')
    msc_spec = Spectrum.from_representation(msc_basis, msc, 'dirichlet', 'dirichlet')

    E1s_sc.append(sc_spec.energies[0]) ; E1s_msc.append(msc_spec.energies[0])
    E2s_sc.append(sc_spec.energies[1]) ; E2s_msc.append(msc_spec.energies[1])
    E3s_sc.append(sc_spec.energies[2]) ; E3s_msc.append(msc_spec.energies[2])

    break

    del sc_basis, msc_basis, sc_spec, msc_spec

    print(f'Alpha {_a:.2e} done...')



In [ ]:
N//2

In [ ]:
b1 = np.logspace( 0, np.log10(rM + 1), N//2 ) - 1


In [ ]:
alpha1 = np.logspace(-1, -10, 10)
alpha2 = np.logspace(-1, 1, 20)
alphas = np.unique(np.concatenate((alpha1[::-1], alpha2)))

E1s_sc = []
E2s_sc = []
E3s_sc = []

E1s_msc = []
E2s_msc = []
E3s_msc = []


box_size = 5000

for i, alpha in enumerate(alphas):
    sc  = SoftCoulomb(alpha=alpha)
    msc = MorseSoftCoulomb(alpha=alpha)
    
    rm = -3.5*alpha ; rM = 100 ; N = 500 - 2*i  ; k = 9

    msc_bps   = np.logspace( 0, np.log10(rM + 1), N ) - 1 + rm
    msc_basis = BSplineBasis(msc_bps, k)
    
    b1       = np.logspace( 0, np.log10(rM + 1), N//2 ) - 1
    sc_bps   = np.unique(np.concatenate((-b1[::-1], b1)))
    sc_basis = BSplineBasis(sc_bps, k)

    msc_spec = Spectrum.from_representation(msc_basis, msc, 'dirichlet', 'dirichlet')
    sc_spec  = Spectrum.from_representation(sc_basis, sc, 'dirichlet', 'dirichlet')


    E1s_sc.append(sc_spec.energies[0]) ; E1s_msc.append(msc_spec.energies[0])
    E2s_sc.append(sc_spec.energies[1]) ; E2s_msc.append(msc_spec.energies[1])
    E3s_sc.append(sc_spec.energies[2]) ; E3s_msc.append(msc_spec.energies[2])

    break

    del msc_basis, msc_spec

    print(f'Alpha {alpha:.2e} done...'
          f'E1 = {E1s_msc[-1]:.5f} '
          f'E2 = {E2s_msc[-1]:.5f} '
          f'E3 = {E3s_msc[-1]:.5f} '
          )



In [ ]:
rs = np.linspace(-20, 20, 1000)
plt.plot(rs, sc_spec.wavefunction(2)(rs))
plt.plot(rs, msc_spec.wavefunction(0)(rs))

In [ ]:
sc_spec.energies[2]

In [ ]:
from emerald.quantum.coulomb_unperturbed import C_energy_levels

In [ ]:
fig, ax = figure()

Es_msc = {
    0: E1s_msc,
    1: E2s_msc,
    2: E3s_msc,
}

Es_sc = {
    0: E1s_sc,
    1: E2s_sc,
    2: E3s_sc,
}

for i in range(3):
    ax.plot(alphas, Es_msc[i], label=f"$E_{i+1}$", c=Perc5[i+1])
    ax.plot(alphas, Es_sc[i], ls='dotted', c=Perc5[i+1])
    plt.axhline(C_energy_levels(i+1), alphas[0], alphas[-1], ls='dashed', c='black', alpha=0.5)

ax.set_xlabel(r'$\alpha$')
ax.set_ylabel('$E_i$', rotation=0, labelpad=10)


ax.set_xlim(alphas[0], alphas[-1])
ax.set_ylim(-1.5, -1.e-3)
ax.legend()

ax.set_xscale('log')
# ax.set_yscale('symlog')
apply_axis_style(ax)

axr = ax.twinx()
axr.set_ylabel(
    r"$\mathrm{placeholder}$",
    color="none",
    rotation=0
)
axr.tick_params(
    right=False,
    labelright=False
)
axr.spines["right"].set_visible(False)

In [ ]:
i = 6
alphas[i], E1s_sc[i]

In [ ]:
alphas = np.logspace(-4, 0.3, 100)
rmins = get_r_exact(alphas, 1)
rmins = get_r_approx(alphas, 1)
plt.plot(alphas, rmins)
plt.show()
Vs = [MorseSoftCoulomb(alpha)(get_r_exact(alpha, 10)) for alpha in alphas]

plt.plot(alphas, Vs)

In [ ]:
print(10*2*box_size/(1.e-4))
print(1.e9)